<div align="center">

# Day 04: Intelligence Layer of the Twin

**Digital Twin with Python (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/digital-twin-veltech-NB-lecture/tree/main/days/day-04) · [Lecture page](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html) · [Interactive lab](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lab.html) · [PDF notes](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/Day04_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Nothing needs to be installed.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, asdict
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.6), 'axes.grid': True, 'grid.alpha': 0.3})

def check(name, value, expected, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback."""
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok



# The asset: a 48 V lithium-ion battery pack under a duty cycle.
#
#   states   SoC    state of charge, 0 to 1
#            T      cell temperature in degrees Celsius
#            Q      usable capacity in Ah, fades with use
#            R      internal resistance in ohm, grows as capacity fades
#   inputs   I(t)   current demanded by the load in A, positive when discharging
#            u      cooling-fan duty, 0 to 1
#            T_amb  ambient temperature in degrees Celsius
#
# The state of charge changes over minutes, the temperature over tens of minutes and
# the capacity over months.

@dataclass
class PackParams:
    Q0: float = 60.0        # Ah, capacity when new
    R0: float = 0.045       # Ohm, internal resistance when new
    m_cp: float = 9000.0    # J/K, thermal mass of the pack
    hA0: float = 2.2        # W/K, passive convection
    hA1: float = 12.0       # W/K, extra convection at full fan
    k_age: float = 5.5e-8   # Ah lost per A-second at the reference temperature
    Ea_R: float = 3800.0    # K, Arrhenius slope of the ageing law
    T_ref: float = 298.15   # K
    fan_W: float = 25.0     # W, electrical cost of the fan at full duty

    def as_vector(self):
        return np.array([self.R0, self.m_cp, self.hA0, self.hA1])


def pack_derivatives(t, x, I, u, T_amb, p: PackParams):
    """dx/dt for x = [SoC, T, Q, R]. Pure physics, no data, no noise."""
    SoC, T, Q, R = x
    T_K = T + 273.15
    P_gen = I ** 2 * R                                    # ohmic heating, W
    P_cool = (p.hA0 + p.hA1 * u) * (T - T_amb)            # convective loss, W
    arrh = np.exp(-p.Ea_R * (1.0 / T_K - 1.0 / p.T_ref))  # ageing accelerates with heat
    dSoC = -I / (3600.0 * max(Q, 1e-6))
    dT = (P_gen - P_cool) / p.m_cp
    dQ = -p.k_age * abs(I) * arrh
    dR = p.R0 * 0.9 * (p.Q0 - Q) / p.Q0 * 1e-3            # resistance tracks capacity fade
    return np.array([dSoC, dT, dQ, dR])


def duty_cycle(t, kind="urban", seed=RANDOM_STATE):
    """Current demand in amperes as a function of time in seconds."""
    rng = np.random.default_rng(seed)
    if kind == "urban":       # stop-start: bursts of traction, regenerative braking
        base = 14 + 26 * (np.sin(2 * np.pi * t / 420.0) > 0.15)
        regen = -18 * (np.sin(2 * np.pi * t / 420.0 + 0.9) > 0.93)
        return base + regen
    if kind == "highway":     # steady high load
        return 40 + 6 * np.sin(2 * np.pi * t / 1800.0)
    if kind == "idle":
        return 8 + 0 * t
    raise ValueError(kind)



def rk4_step(fun, t, x, dt, *args):
    k1 = fun(t, x, *args)
    k2 = fun(t + dt / 2, x + dt / 2 * k1, *args)
    k3 = fun(t + dt / 2, x + dt / 2 * k2, *args)
    k4 = fun(t + dt, x + dt * k3, *args)
    return x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def simulate(p: PackParams, T_end=3600.0, dt=1.0, x0=None, cycle="urban",
             controller=None, T_amb=25.0, seed=RANDOM_STATE):
    """Roll the physical asset forward. `controller(t, x) -> fan duty` closes the loop."""
    x = np.array([0.95, T_amb + 1.0, p.Q0, p.R0]) if x0 is None else np.array(x0, float)
    n = int(T_end / dt)
    ts = np.arange(n + 1) * dt
    X = np.zeros((n + 1, 4)); X[0] = x
    U = np.zeros(n + 1); I_log = np.zeros(n + 1)
    for k in range(n):
        t = ts[k]
        I = float(duty_cycle(t, cycle, seed))
        u = 0.0 if controller is None else float(np.clip(controller(t, x), 0, 1))
        U[k] = u; I_log[k] = I
        x = rk4_step(pack_derivatives, t, x, dt, I, u, T_amb, p)
        x[0] = np.clip(x[0], 0.0, 1.0)
        X[k + 1] = x
    U[-1] = U[-2]; I_log[-1] = I_log[-2]
    return pd.DataFrame({"t": ts, "SoC": X[:, 0], "T": X[:, 1], "Q": X[:, 2],
                         "R": X[:, 3], "u": U, "I": I_log})



@dataclass
class SensorSpec:
    """Turns a true value into a reading: noise, bias, drift, quantisation, dropout and latency."""
    noise_sd: float = 0.0       # gaussian noise, in sensor units
    bias: float = 0.0           # constant offset
    drift_per_hour: float = 0.0 # slow calibration drift
    quantum: float = 0.0        # ADC quantisation step, 0 = none
    dropout: float = 0.0        # probability a sample is lost
    latency_s: float = 0.0      # transport delay


def read_sensor(t, true_values, spec: SensorSpec, rng):
    v = np.asarray(true_values, float).copy()
    v = v + spec.bias + spec.drift_per_hour * (t / 3600.0)
    if spec.noise_sd:
        v = v + rng.normal(0, spec.noise_sd, v.shape)
    if spec.quantum:
        v = np.round(v / spec.quantum) * spec.quantum
    if spec.dropout:
        v = np.where(rng.random(v.shape) < spec.dropout, np.nan, v)
    return v


print('pack model, integrator and sensors ready')

## Python step 4: Plots and the scikit-learn interface

Matplotlib draws figures from arrays &#91;[5](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#ref-5)&#93;, and scikit-learn gives every model the same three methods: `fit`, `predict` and `score` &#91;[6](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#ref-6)&#93;. This step simulates forty steady operating points of the pack, plots them and fits a model that predicts the peak temperature.

### Data and a labelled scatter plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
current = rng.uniform(20, 60, 40)                 # A
ambient = rng.uniform(20, 40, 40)                 # degrees C
peak = ambient + current ** 2 * 0.045 / 14.2 + rng.normal(0, 0.3, 40)   # steady state, fan at full duty
fig, ax = plt.subplots(figsize=(6, 3.4))
sc = ax.scatter(current, peak, c=ambient, cmap="viridis")
ax.set_xlabel("current (A)"); ax.set_ylabel("peak temperature (°C)")
fig.colorbar(sc, label="ambient (°C)")
plt.show()
print("mean peak:", round(float(peak.mean()), 2), "°C")

`plt.subplots` creates a figure and an axes object, and the methods of `ax` draw on it and label it. In `scatter`, the argument `c` colours each point by a third variable, and `colorbar` explains the colours. The peak temperature follows from the energy balance at steady state with the fan at full duty, plus measurement noise. It is the ambient temperature plus $I^2 R_0 / (hA_0 + hA_1)$, the steady-state formula of the lecture with the current $I$ and the true values $R_0 = 0.045$ ohm and $hA_0 + hA_1 = 14.2$ W/K. The noise has a standard deviation of 0.3 °C.

### Training and test data

In [ ]:
from sklearn.model_selection import train_test_split
X = np.column_stack([current ** 2, ambient])      # features: current squared, ambient
X_train, X_test, y_train, y_test = train_test_split(X, peak, test_size=0.25, random_state=0)
print(X_train.shape, X_test.shape)

`np.column_stack` places arrays side by side as the columns of a feature matrix. `train_test_split` keeps a quarter of the points aside for testing, so that the model is judged on data it has not seen. The first feature is the square of the current, because the ohmic heat is proportional to it.

### Fit, predict, measure

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
model = LinearRegression().fit(X_train, y_train)
print("coefficients:", model.coef_.round(4), "intercept:", round(float(model.intercept_), 2))
print("test MAE:", round(float(mean_absolute_error(y_test, model.predict(X_test))), 3), "°C")
print("prediction for 50 A at 30 °C:", round(float(model.predict([[50 ** 2, 30]])[0]), 2), "°C")

`fit` learns the coefficients from the training data and returns the model, so it can be chained after the constructor. `predict` applies the model to new rows. The coefficient of the ambient temperature is close to 1, and the coefficient of the squared current is close to the physical value 0.045 / 14.2, about 0.0032. The learned model has recovered the physics. The test MAE is the mean absolute error on the test data: the average size of the differences between the predicted and the measured peak temperatures, in °C. The third method of the interface, `score`, returns for such a model the coefficient of determination, a number that is 1 for perfect predictions and 0 for a model that always predicts the mean. Here `model.score(X_test, y_test)` gives 0.998.

### The value of a physical feature

In [ ]:
X_raw = np.column_stack([current, ambient])       # the current itself instead of its square
Xr_train, Xr_test, _, _ = train_test_split(X_raw, peak, test_size=0.25, random_state=0)
raw = LinearRegression().fit(Xr_train, y_train)
print("test MAE with the raw current:", round(float(mean_absolute_error(y_test, raw.predict(Xr_test))), 3), "°C")

With the raw current as a feature, the test error of the same linear model rises from 0.242 to 0.364 °C, because the true relation is quadratic. A feature taken from physics often helps a simple model more than a more complex model would, which is the idea behind the hybrid models of the lecture.

**Quick check.** Why is the model judged on `X_test` rather than on `X_train`?

<details><summary>Answer</summary>

Because an error measured on the training data says how well the model memorised them, not how well it predicts data it has not seen.

</details>

### Exercises for Python step 4

**Exercise 4.1.** How many points are in the test set? Store the number in `n_test`.

In [ ]:
n_test = None   # your answer

check("Exercise 4.1", n_test, 10)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_test = len(y_test)

check("Exercise 4.1", n_test, 10)

**Exercise 4.2.** Use `model` to predict the peak temperature for 50 A at 30 °C ambient, round it to two decimals and store it in `p50`. Remember that the first feature is the square of the current.

In [ ]:
p50 = None   # your answer

check("Exercise 4.2", p50, 37.93)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p50 = round(float(model.predict([[50 ** 2, 30]])[0]), 2)

check("Exercise 4.2", p50, 37.93)

**Exercise 4.3.** Store the test MAE of the model with the raw current, rounded to three decimals, in `mae_raw`.

In [ ]:
mae_raw = None   # your answer

check("Exercise 4.3", mae_raw, 0.364)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
mae_raw = round(float(mean_absolute_error(y_test, raw.predict(Xr_test))), 3)

check("Exercise 4.3", mae_raw, 0.364)

## 1. The calibrated core

*Lecture: [A calibrated core](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#a-calibrated-core)*

**From the formulas to the code.** `CAL` holds the calibrated parameters of the lecture table, and `PackParams()` without arguments gives the true values of the simulated pack. `simulate` advances the four equations of `pack_derivatives` with `rk4_step`, the Runge-Kutta method of fourth order of Day 2. Here $\Delta t = 1$ s, so `T_end=7200` means 7200 steps. In `pack_derivatives`, `P_gen` is the heat $I^2 R$, `P_cool` the cooling power $(hA_0 + hA_1 u)(T - T_{\mathrm{amb}})$ and `arrh` the Arrhenius factor of the ageing law. The code guards the division by the capacity with `max(Q, 1e-6)`, and `simulate` keeps the state of charge (SoC) between 0 and 1 with `np.clip`.

In [ ]:
import time
TRUE = PackParams()
CAL = PackParams(R0=0.0456, m_cp=9180.0, hA0=2.193, hA1=12.23)   # Day 3: m_cp measured, three parameters fitted
print("calibrated parameters carried over from Day 3:", CAL.as_vector().round(4))
t0 = time.time(); base = simulate(CAL, T_end=7200, dt=1.0, cycle="highway", T_amb=35.0)
print(f"one full 2-hour simulation: {time.time()-t0:.3f} s at dt = 1 s")

## 2. A Gaussian-process surrogate &#91;[1](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#ref-1)&#93;

*Lecture: [Surrogate models and their limits](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#surrogate-models-and-their-limits)*

**From the formulas to the code.** `peak_T` is $T_{\max}(q)$: It simulates one hour with $\Delta t = 5$ s and keeps the largest temperature in `best`. `Xd` holds the $n = 120$ training queries inside the box from `LO` to `HI`, and `yd` their outputs. `poly` is the quadratic baseline: `StandardScaler` standardises the inputs, `PolynomialFeatures(2)` forms the squares and products, and `Ridge(alpha=1e-3)` is the least-squares fit with a small penalty. In `kern`, `ConstantKernel` is $\sigma_f^2$, `Matern(length_scale=[8., 12., .4], nu=2.5)` is the Matérn kernel with the starting length scales and `WhiteKernel` is the noise variance $\sigma_n^2$. `normalize_y=True` subtracts the mean of the outputs and divides them by their standard deviation before the fit, and `alpha=1e-6` adds a tiny number to the diagonal of $\mathbf{K}$ for numerical safety. The printed RMSE, the root mean square error, is `np.sqrt(np.mean(e**2))`, and the break-even point is `t_design / (t_sim - t_gp)`. GP in the names and printed lines stands for Gaussian process.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

def peak_T(T_amb, I_scale, u, p=CAL, T_end=3600.0, dt=5.0):
    """The expensive reference: one run of the simulator, which returns the maximum temperature."""
    x = np.array([0.9, T_amb + 1.0, p.Q0, p.R0]); best = x[1]
    for k in range(int(T_end / dt)):
        t = k * dt
        I = I_scale * float(duty_cycle(t, "highway")) / 40.0
        x = rk4_step(pack_derivatives, t, x, dt, I, u, T_amb, p)
        x[0] = np.clip(x[0], 0, 1); best = max(best, x[1])
    return best

rng = np.random.default_rng(RANDOM_STATE)
N_TRAIN = 120
LO = np.array([20.0, 20.0, 0.0]); HI = np.array([40.0, 55.0, 1.0])     # the sampled box
Xd = LO + rng.random((N_TRAIN, 3)) * (HI - LO)
t0 = time.time()
yd = np.array([peak_T(*row) for row in Xd])
t_design = time.time() - t0
print(f"design of experiments: {N_TRAIN} simulations in {t_design:.1f} s "
      f"({t_design/N_TRAIN*1000:.0f} ms each)")

poly = make_pipeline(StandardScaler(), PolynomialFeatures(2), Ridge(alpha=1e-3)).fit(Xd, yd)
kern = ConstantKernel(1.0) * Matern(length_scale=[8., 12., .4], nu=2.5) + WhiteKernel(1e-3)
gp = GaussianProcessRegressor(kernel=kern, normalize_y=True, alpha=1e-6).fit(Xd, yd)

Xv = LO + rng.random((40, 3)) * (HI - LO)
yv = np.array([peak_T(*row) for row in Xv])
for nm, m in (("quadratic", poly), ("gaussian process", gp)):
    e = m.predict(Xv) - yv
    print(f"  {nm:17s} RMSE {np.sqrt(np.mean(e**2)):.3f} C   max |err| {np.abs(e).max():.3f} C")

t0 = time.time(); [peak_T(*Xv[0]) for _ in range(20)]; t_sim = (time.time() - t0) / 20
t0 = time.time(); [gp.predict(Xv[:1]) for _ in range(20)]; t_gp = (time.time() - t0) / 20
print(f"\nsimulator {t_sim*1000:8.2f} ms per query")
print(f"surrogate {t_gp*1000:8.2f} ms per query   -> {t_sim/t_gp:.0f}x faster")
print(f"break-even: the {t_design:.0f} s design of experiments is recovered after "
      f"{t_design/(t_sim - t_gp):.0f} queries")

## 3. Outside the box

*Lecture: [Surrogate models and their limits](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#surrogate-models-and-their-limits)*

**From the formulas to the code.** The three regions are boxes around the centre `(LO + HI) / 2` whose half-widths are multiplied by 1.0, 1.25 and 1.6. `np.clip` keeps the fan duty between 0 and 1, so only the ambient temperature and the current leave the training box. `gp.predict(..., return_std=True)` returns the prediction $\mu(q)$ and the standard deviation $s(q)$, printed as `GP std`. The value of scikit-learn adds the small noise variance $\sigma_n^2$ to $s(q)^2$. The guard of Exercise A is the test $\mathrm{LO}_j \le q_j \le \mathrm{HI}_j$ for all three inputs of the query `x`.

In [ ]:
# The accuracy of the surrogates outside the sampled box.
probe = []
for factor, label in ((1.0, "inside the box"), (1.25, "25% outside"), (1.6, "60% outside")):
    centre = (LO + HI) / 2; half = (HI - LO) / 2 * factor
    P_ = np.clip(centre + (rng.random((25, 3)) * 2 - 1) * half,
                 [5, 10, 0], [70, 90, 1])
    yt = np.array([peak_T(*r) for r in P_])
    for nm, m in (("quadratic", poly), ("gaussian process", gp)):
        e = m.predict(P_) - yt
        probe.append(dict(region=label, model=nm, rmse=np.sqrt(np.mean(e ** 2)),
                          max_err=np.abs(e).max()))
pv = pd.DataFrame(probe).pivot(index="region", columns="model", values=["rmse", "max_err"])
print(pv.round(2).to_string())

# The Gaussian process (GP) reports a larger standard deviation outside the box. The polynomial reports none.
mu, sd = gp.predict(np.array([[30., 38., .5], [30., 80., .5], [60., 90., .0]]), return_std=True)
for row, m_, s_ in zip(["centre of box", "current far outside", "far outside on two axes"], mu, sd):
    print(f"  {row:26s} prediction {m_:7.2f} C   GP std {s_:6.2f} C")

**Exercise A.** Write a function `in_box(x)` that returns `True` only if every input of the query `x` lies within the bounds `LO` and `HI` of the design. Store `in_box(np.array([30.0, 70.0, 0.5]))` in `ok_query`; the second input, 70, lies outside its range.

In [ ]:
ok_query = None   # your answer

check("Exercise A", ok_query, False)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def in_box(x):
    return bool(np.all((np.asarray(x) >= LO) & (np.asarray(x) <= HI)))

ok_query = in_box(np.array([30.0, 70.0, 0.5]))

check("Exercise A", ok_query, False)

## 4. A hybrid model &#91;[2](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#ref-2)&#93;

*Lecture: [Hybrid models](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#hybrid-models)*

**From the formulas to the code.** `real_derivatives` replaces the resistance by $R_{\mathrm{eff}} = R\,[1 + 0.012\,(T - 25)]$ and plays the pack of this experiment. The column `T_phys` is the prediction of the model without this effect, and `FEATS` lists the five inputs $\phi$. `ml` is $g_{\mathrm{ML}}$, fitted to the temperature `tr["T"]`. `hyb` is $g$, fitted to the difference `tr["T"] - tr["T_phys"]`, so the hybrid prediction is `df.T_phys + hyb.predict(df[FEATS])`. `score` computes the root mean square error (RMSE) of the three models, and `degradation_x` is the error in the heatwave divided by the error on the mild days. `T_phys` is stored before each step of 10 s and `T` after it, so the two columns are one step apart. ML in the names and labels stands for machine learning.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

def real_derivatives(t, x, I, u, T_amb, p):
    """The physical pack: Its resistance rises with temperature, which the model does not include."""
    SoC, T, Q, R = x
    R_eff = R * (1.0 + 0.012 * (T - 25.0))          # <-- the missing physics
    return pack_derivatives(t, [SoC, T, Q, R_eff], I, u, T_amb, p)

def run_real(T_amb, u, T_end=3600.0, dt=10.0, p=CAL):
    x = np.array([0.9, T_amb + 1.0, p.Q0, p.R0]); out = []
    for k in range(int(T_end / dt)):
        t = k * dt; I = float(duty_cycle(t, "highway"))
        x = rk4_step(real_derivatives, t, x, dt, I, u, T_amb, p)
        x[0] = np.clip(x[0], 0, 1)
        out.append(dict(t=t, T_amb=T_amb, u=u, I=I, T=x[1], SoC=x[0]))
    return pd.DataFrame(out)

TRAIN_AMB = [22.0, 26.0, 30.0]        # the training data cover mild days only
TEST_AMB = [34.0, 40.0, 46.0]         # a heatwave

def build_dataset(ambs):
    rows = []
    for T_amb in ambs:
        for u in (0.0, 0.5, 1.0):
            real = run_real(T_amb, u)
            phys = np.zeros(len(real)); x = np.array([0.9, T_amb + 1.0, CAL.Q0, CAL.R0])
            for k in range(len(real)):
                phys[k] = x[1]
                x = rk4_step(pack_derivatives, real.t.values[k], x, 10.0,
                             real.I.values[k], u, T_amb, CAL)
                x[0] = np.clip(x[0], 0, 1)
            real["T_phys"] = phys
            rows.append(real)
    return pd.concat(rows, ignore_index=True)

tr = build_dataset(TRAIN_AMB); te = build_dataset(TEST_AMB)
FEATS = ["t", "T_amb", "u", "I", "T_phys"]

ml = GradientBoostingRegressor(random_state=RANDOM_STATE).fit(tr[FEATS], tr["T"])
hyb = GradientBoostingRegressor(random_state=RANDOM_STATE).fit(tr[FEATS], tr["T"] - tr["T_phys"])

def score(df):
    return dict(
        physics=np.sqrt(np.mean((df.T_phys - df["T"]) ** 2)),
        ml_only=np.sqrt(np.mean((ml.predict(df[FEATS]) - df["T"]) ** 2)),
        hybrid=np.sqrt(np.mean((df.T_phys + hyb.predict(df[FEATS]) - df["T"]) ** 2)))

res = pd.DataFrame({"train (22-30 C)": score(tr), "test (34-46 C)": score(te)})
res["degradation_x"] = res["test (34-46 C)"] / res["train (22-30 C)"]
res = res.sort_values("test (34-46 C)")
print("RMSE against the real pack [C]\n")
print(res.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for a, (df, ttl) in zip(ax, ((tr[tr.T_amb == 26], "Training day, 26 C ambient"),
                             (te[te.T_amb == 46], "Heatwave, 46 C ambient (unseen)"))):
    d = df[df.u == 0.5]
    a.plot(d.t / 60, d["T"], lw=2.2, color="#111111", label="real pack")
    a.plot(d.t / 60, d.T_phys, lw=1.3, ls="--", color="#c0392b", label="physics only")
    a.plot(d.t / 60, ml.predict(d[FEATS]), lw=1.3, color="#f39c12", label="ML only")
    a.plot(d.t / 60, d.T_phys + hyb.predict(d[FEATS]), lw=1.3, color="#27ae60", label="hybrid")
    a.set_title(ttl, fontsize=9); a.set_xlabel("time [min]")
ax[0].set_ylabel("T [C]"); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("RMSE on the heatwave:")
for k, v in score(te).items():
    print(f"   {k:9s} RMSE {v:6.3f} C")

## 5. Residuals and a cumulative sum &#91;[3](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#ref-3)&#93;

*Lecture: [Detecting faults](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#detecting-faults)*

**From the formulas to the code.** `inject_fault` simulates the pack and switches to changed parameters once `t >= onset`. `twin_residual` runs the unchanged model with the same current and fan duty and returns the residual $r_i = y_i - \hat{y}_i$ as `df["T"].values - np.array(pred)`. In `cusum`, the line `s[i] = max(0.0, s[i - 1] + r[i] - k)` is the cumulative sum (CUSUM) with the allowance `k=0.15` and the decision limit `h=4.0`, and `alarm` is the first index with `s[i] > h`. The loop starts at `i = 1`, so the first residual is not used. The prediction is stored before each step and the reading after it. A healthy residual is therefore not exactly zero: It equals the change of the temperature within one sample, at most 0.07 °C, which the allowance absorbs. The column `raw_threshold_s` is the fixed limit, the first time at which the temperature exceeds 46 °C.

In [ ]:
def inject_fault(kind, T_end=5400.0, dt=10.0, T_amb=35.0, onset=3000.0):
    """Three faults that a threshold on the raw temperature detects late."""
    x = np.array([0.9, T_amb + 1.0, CAL.Q0, CAL.R0]); rows = []
    for k in range(int(T_end / dt)):
        t = k * dt; I = float(duty_cycle(t, "highway")); u = 0.5
        p = CAL
        if t >= onset:
            if kind == "fan degradation":
                p = PackParams(R0=CAL.R0, m_cp=CAL.m_cp, hA0=CAL.hA0, hA1=CAL.hA1 * 0.35)
            elif kind == "cell resistance rise":
                p = PackParams(R0=CAL.R0 * 1.30, m_cp=CAL.m_cp, hA0=CAL.hA0, hA1=CAL.hA1)
            elif kind == "coolant blockage":
                p = PackParams(R0=CAL.R0, m_cp=CAL.m_cp, hA0=CAL.hA0 * 0.4, hA1=CAL.hA1 * 0.6)
        x = rk4_step(pack_derivatives, t, x, dt, I, u, T_amb, p)
        x[0] = np.clip(x[0], 0, 1)
        rows.append(dict(t=t, I=I, u=u, T=x[1]))
    return pd.DataFrame(rows)

def twin_residual(df, T_amb=35.0):
    x = np.array([0.9, T_amb + 1.0, CAL.Q0, CAL.R0]); pred = []
    for k in range(len(df)):
        pred.append(x[1])
        x = rk4_step(pack_derivatives, df.t.values[k], x, 10.0,
                     df.I.values[k], df.u.values[k], T_amb, CAL)
        x[0] = np.clip(x[0], 0, 1)
    return df["T"].values - np.array(pred)

def cusum(r, k=0.15, h=4.0):
    """Detects a small persistent shift far sooner than a threshold on the raw residual."""
    s = np.zeros(len(r)); alarm = None
    for i in range(1, len(r)):
        s[i] = max(0.0, s[i - 1] + r[i] - k)
        if alarm is None and s[i] > h:
            alarm = i
    return s, alarm

rows = []
for kind in ("fan degradation", "cell resistance rise", "coolant blockage"):
    d = inject_fault(kind); r = twin_residual(d)
    s, a = cusum(r)
    thr = np.argmax(d["T"].values > 46.0) if (d["T"].values > 46.0).any() else None
    rows.append(dict(fault=kind,
                     cusum_alarm_s=d.t.values[a] if a else None,
                     raw_threshold_s=d.t.values[thr] if thr else None,
                     warning_gained_min=((d.t.values[thr] - d.t.values[a]) / 60)
                     if (a and thr) else np.nan,
                     peak_T=d["T"].max()))
det = pd.DataFrame(rows)
print(det.round(1).to_string(index=False))

**Exercise B.** How many minutes of warning does the cumulative sum gain over the fixed limit for the fan degradation? Store the value from the table `det` in `gain_fan`.

In [ ]:
gain_fan = None   # your answer

check("Exercise B", gain_fan, float(det.set_index("fault").loc["fan degradation", "warning_gained_min"]))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
gain_fan = float(det.set_index("fault").loc["fan degradation", "warning_gained_min"])

check("Exercise B", gain_fan, float(det.set_index("fault").loc["fan degradation", "warning_gained_min"]))

## 6. Remaining useful life &#91;[4](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#ref-4)&#93;

*Lecture: [Remaining useful life](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#remaining-useful-life)*

**From the formulas to the code.** `EOL` is the end-of-life level $Q_{\mathrm{EOL}} = 0.8\,Q_0$. In `rul_estimate`, `np.polyfit(x[idx], y[idx], 1)` returns the slope $b$ and the intercept $a$ of the line, in this order, and `(EOL - a) / b - x[-1]` is the remaining useful life (RUL), $n_{\mathrm{EOL}} - n_{\mathrm{last}}$. `idx = rng.integers(0, len(x), len(x))` draws one bootstrap resample with replacement, and `n_boot=200` sets the number of resamples. A resample whose line does not fall, `b >= 0`, gets an infinite life. The function returns the median and the 5th and 95th percentiles. A remaining life is counted from the latest trip of its history, trip 89 or trip 179. The last lines of the cell therefore add this number: `eol180` is the end-of-life trip predicted after 180 trips, 750, and `eol_lo90` and `eol_hi90` are the ends of the interval predicted after 90 trips, 781 and 817.

In [ ]:
def fleet_history(n_trips=180, seed=RANDOM_STATE):
    """Capacity measured once per trip, noisily, with the duty cycle varying trip to trip."""
    rng = np.random.default_rng(seed)
    Q = CAL.Q0; rows = []
    for trip in range(n_trips):
        T_amb = float(np.clip(rng.normal(32, 6), 15, 46))
        cyc = rng.choice(["urban", "highway"], p=[0.6, 0.4])
        p = PackParams(Q0=Q, R0=CAL.R0, m_cp=CAL.m_cp, hA0=CAL.hA0, hA1=CAL.hA1)
        d = simulate(p, T_end=5400, dt=20.0, cycle=cyc, T_amb=T_amb,
                     controller=lambda t, x: float(np.clip((x[1] - 42) / 6, 0, 1)))
        Q = float(d.Q.iloc[-1])
        rows.append(dict(trip=trip, Q_true=Q, Q_meas=Q + rng.normal(0, 0.05),
                         T_amb=T_amb, cycle=cyc, T_max=float(d["T"].max())))
    return pd.DataFrame(rows)

hist = fleet_history()
EOL = 0.80 * CAL.Q0
print(f"end of life threshold: {EOL:.1f} Ah   after {len(hist)} trips the pack is at "
      f"{hist.Q_meas.iloc[-1]:.2f} Ah ({100*hist.Q_meas.iloc[-1]/CAL.Q0:.1f}%)")

def rul_estimate(hist, n_boot=200, seed=RANDOM_STATE):
    """Linear extrapolation of the fade trend, with a bootstrap interval."""
    rng = np.random.default_rng(seed)
    x, y = hist.trip.values.astype(float), hist.Q_meas.values
    out = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(x), len(x))
        b, a = np.polyfit(x[idx], y[idx], 1)
        out.append((EOL - a) / b - x[-1] if b < 0 else np.inf)
    out = np.array(out)
    return float(np.median(out)), float(np.percentile(out, 5)), float(np.percentile(out, 95))

ests = {}
for n in (40, 90, 180):
    med, lo, hi = rul_estimate(hist.iloc[:n])
    ests[n] = (med, lo, hi)
    print(f"after {n:3d} trips: RUL {med:6.0f} trips   90% interval [{lo:.0f}, {hi:.0f}]"
          f"   width {hi-lo:5.0f}")

m90, lo90, hi90 = ests[90]; m180, _, _ = ests[180]
# remaining lives are counted from the latest trip: trip 179 and trip 89
eol180 = 179 + m180; eol_lo90, eol_hi90 = 89 + lo90, 89 + hi90
inside = eol_lo90 <= eol180 <= eol_hi90
print(f"\nthe end of life predicted after 180 trips (trip {eol180:.0f}) falls inside the 90% interval")
print(f"predicted after 90 trips [trip {eol_lo90:.0f}, trip {eol_hi90:.0f}]: {inside}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].plot(hist.trip, hist.Q_meas, ".", ms=3, color="#7f8c8d", label="measured")
ax[0].plot(hist.trip, hist.Q_true, lw=1.6, color="#2980b9", label="true capacity")
b, a_ = np.polyfit(hist.trip, hist.Q_meas, 1)
xf = np.arange(0, 900)
ax[0].plot(xf, a_ + b * xf, ls="--", lw=1, color="#c0392b", label="extrapolated trend")
ax[0].axhline(EOL, ls=":", color="#111111"); ax[0].set_xlim(0, 900)
ax[0].set_xlabel("trip"); ax[0].set_ylabel("capacity [Ah]"); ax[0].legend(fontsize=8)
ax[0].set_title("Health indicator and its extrapolation")

ns = np.arange(20, 181, 10)
band = np.array([rul_estimate(hist.iloc[:n], n_boot=120) for n in ns])
ax[1].fill_between(ns, band[:, 1], band[:, 2], alpha=.25, color="#27ae60")
ax[1].plot(ns, band[:, 0], color="#27ae60")
ax[1].set_xlabel("trips of history used"); ax[1].set_ylabel("estimated RUL [trips]")
ax[1].set_title("The interval narrows as evidence accumulates")
plt.tight_layout(); plt.show()

## 7. What-if scenarios and a twin in the loop

*Lecture: [What-if scenarios and control](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#what-if-scenarios-and-control)*

**From the formulas to the code.** In `scenario`, the `controller` is the fixed rule $u = \mathrm{clip}((T - T_{\mathrm{on}}) / 6, 0, 1)$ with `fan_on` as $T_{\mathrm{on}}$, and `trips_to_EOL` is $n_{\mathrm{EOL}}$. `fan_energy` adds up $u_k\,\Delta t\,P_{\mathrm{fan}}$ divided by $3.6 \times 10^6$, the fan energy in kilowatt-hours (kWh). `rollout_cost` is the cost $J(u)$ of the planner with the weights `w_hot`, `w_fan` and `w_age`, and the capacity is the third entry of the state, `x[2]`. In `run_controller`, `grid = np.linspace(0, 1, 6)` holds the six candidate duties and `np.argmin` picks the cheapest one at every step. The column `objective` is $J_{\mathrm{trip}}$ with the weights 60, 500 and 2, and `capacity_lost_mAh` is the lost capacity in milliampere-hours (mAh). Other weights for the ranking mean other numbers in that line. The planner follows them only when `w_hot`, `w_fan` and `w_age` are changed as well.

In [ ]:
def scenario(name, cycle="highway", T_amb=35.0, fan_on=42.0, trips=120):
    Q = CAL.Q0; hot_s = 0.0; fan_energy = 0.0
    for _ in range(trips):
        p = PackParams(Q0=Q, R0=CAL.R0, m_cp=CAL.m_cp, hA0=CAL.hA0, hA1=CAL.hA1)
        d = simulate(p, T_end=5400, dt=20.0, cycle=cycle, T_amb=T_amb,
                     controller=lambda t, x: float(np.clip((x[1] - fan_on) / 6, 0, 1)))
        Q = float(d.Q.iloc[-1])
        hot_s += float((d["T"] > 46).sum()) * 20.0
        fan_energy += float(d.u.sum()) * 20.0 * CAL.fan_W / 3.6e6      # kWh
    return dict(scenario=name, capacity_pct=100 * Q / CAL.Q0,
                hours_over_46C=hot_s / 3600, fan_kWh=fan_energy,
                trips_to_EOL=0.20 * CAL.Q0 / max(CAL.Q0 - Q, 1e-9) * trips)

sc = pd.DataFrame([
    scenario("baseline"),
    scenario("summer (+8 C ambient)", T_amb=43.0),
    scenario("all-urban duty", cycle="urban"),
    scenario("aggressive cooling (fan from 36 C)", fan_on=36.0),
    scenario("cooling disabled", fan_on=99.0),
])
print(sc.round(2).to_string(index=False))

In [ ]:
def rollout_cost(u_candidate, x, t0, horizon=300.0, dt=20.0, T_amb=35.0,
                 w_hot=1.0, w_fan=0.02, w_age=4e4):
    xs = np.array(x, float); cost = 0.0
    for k in range(int(horizon / dt)):
        t = t0 + k * dt
        xs = rk4_step(pack_derivatives, t, xs, dt, float(duty_cycle(t, "highway")),
                      u_candidate, T_amb, CAL)
        cost += w_hot * max(0.0, xs[1] - 44.0) ** 2 * dt / 60
        cost += w_fan * u_candidate * CAL.fan_W * dt / 60
    cost += w_age * (x[2] - xs[2])
    return cost

def run_controller(kind, T_end=5400.0, dt=20.0, T_amb=35.0):
    x = np.array([0.9, T_amb + 1.0, CAL.Q0, CAL.R0]); rows = []
    grid = np.linspace(0, 1, 6)
    for k in range(int(T_end / dt)):
        t = k * dt
        if kind == "fixed rule":
            u = float(np.clip((x[1] - 42) / 6, 0, 1))
        elif kind == "always on":
            u = 1.0
        elif kind == "twin-in-the-loop":
            u = float(grid[int(np.argmin([rollout_cost(g, x, t, T_amb=T_amb) for g in grid]))])
        else:
            u = 0.0
        x = rk4_step(pack_derivatives, t, x, dt, float(duty_cycle(t, "highway")), u, T_amb, CAL)
        x[0] = np.clip(x[0], 0, 1)
        rows.append(dict(t=t, u=u, T=x[1], Q=x[2]))
    d = pd.DataFrame(rows)
    return dict(controller=kind, peak_T=d["T"].max(),
                hours_over_44C=float((d["T"] > 44).sum()) * dt / 3600,
                fan_kWh=float(d.u.sum()) * dt * CAL.fan_W / 3.6e6,
                capacity_lost_mAh=(CAL.Q0 - d.Q.iloc[-1]) * 1000), d

rows, traces = [], {}
for kind in ("no cooling", "always on", "fixed rule", "twin-in-the-loop"):
    r, d = run_controller(kind); rows.append(r); traces[kind] = d
ctrl = pd.DataFrame(rows)
ctrl["objective"] = (ctrl.hours_over_44C * 60 + ctrl.fan_kWh * 500
                     + ctrl.capacity_lost_mAh * 2)
print(ctrl.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3), sharex=True)
for kind, col in (("no cooling", "#7f8c8d"), ("always on", "#2980b9"),
                  ("fixed rule", "#f39c12"), ("twin-in-the-loop", "#27ae60")):
    ax[0].plot(traces[kind].t / 60, traces[kind]["T"], lw=1.4, color=col, label=kind)
    ax[1].plot(traces[kind].t / 60, traces[kind].u, lw=1.1, color=col)
ax[0].axhline(44, ls=":", color="#c0392b")
ax[0].set_ylabel("T [C]"); ax[0].legend(fontsize=8); ax[0].set_xlabel("min")
ax[1].set_ylabel("fan duty"); ax[1].set_xlabel("min")
ax[0].set_title("Temperature"); ax[1].set_title("Control action")
plt.tight_layout(); plt.show()

ctrl_i = ctrl.set_index("controller")
best = ctrl.loc[ctrl.objective.idxmin(), "controller"]
print(f"lowest combined objective: {best}")
print(f"  vs the fixed rule  : {100*(1 - ctrl_i.objective[best]/ctrl_i.objective['fixed rule']):.0f}% better")
print(f"  vs always-on       : {100*(1 - ctrl_i.objective[best]/ctrl_i.objective['always on']):.0f}% better")

**Exercise C.** By how many percent does the twin-in-the-loop controller beat the always-on fan? Compute it from the column `objective` of the table `ctrl` and store it, as a percentage, in `margin`.

In [ ]:
margin = None   # your answer

o_ = ctrl.set_index("controller")["objective"]
check("Exercise C", margin, 100 * (o_["always on"] - o_["twin-in-the-loop"]) / o_["always on"])

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
o = ctrl.set_index("controller")["objective"]
margin = 100 * (o["always on"] - o["twin-in-the-loop"]) / o["always on"]

o_ = ctrl.set_index("controller")["objective"]
check("Exercise C", margin, 100 * (o_["always on"] - o_["twin-in-the-loop"]) / o_["always on"])

## 8. Your asset: watching for a loss of cooling

*Lecture: [Your asset](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#your-asset)*

Change `ASSET` to `server_rack`, `motor` or `greenhouse` and run the cell again.

**From the formulas to the code.** In `watch_asset`, `T` is the temperature of the asset with the faulty conductance `h1`, and `model` is the prediction of the twin with the full `hA1`. Both lines are one explicit Euler step of the thermal model. Every `sync = 30` steps the residual `y - model` is stored and the twin synchronises with `model = y`. `z = res / res[:int(0.3 * len(res))].std()` is the standardised residual $z_j = r_j / s_0$, and `S[j] = max(0.0, S[j - 1] + z[j] - 0.5)` is the cumulative sum. The alarms are the first window with `S > 5.0` and the first with `np.abs(z) > 4.0`. SD on the axis of the plot stands for standard deviation.

In [ ]:
ASSET = "battery"        # choose: battery, server_rack, motor or greenhouse

ASSETS = {
    "battery":     dict(name="battery pack", C=9000.0, hA0=2.2, hA1=12.0, T_amb=35.0, T0=36.0, limit=46.0,
                        horizon=7200.0, dt=5.0, power=lambda t: (40 + 6 * np.sin(2 * np.pi * t / 1800)) ** 2 * 0.045),
    "server_rack": dict(name="server rack", C=1.2e5, hA0=40.0, hA1=400.0, T_amb=24.0, T0=26.0, limit=35.0,
                        horizon=7200.0, dt=10.0, power=lambda t: 3000 + 1500 * np.maximum(0, np.sin(2 * np.pi * t / 3600))),
    "motor":       dict(name="motor winding", C=4000.0, hA0=1.5, hA1=8.0, T_amb=30.0, T0=32.0, limit=90.0,
                        horizon=5400.0, dt=5.0, power=lambda t: (20 + 8 * np.sin(2 * np.pi * t / 1200)) ** 2 * 0.3),
    "greenhouse":  dict(name="greenhouse air", C=3.0e6, hA0=500.0, hA1=4000.0, T_amb=18.0, T0=20.0, limit=32.0,
                        horizon=86400.0, dt=60.0, power=lambda t: 30000 * np.maximum(0, np.sin(2 * np.pi * t / 86400))),
}

def run_asset(a, controller=None, params=None, sensor_sd=0.0, seed=0, T_end=None, dt=None):
    """Explicit Euler for dT/dt = (P(t) - (hA0 + hA1 u)(T - T_amb)) / C; the controller maps a reading to a duty u."""
    p = dict(a); p.update(params or {}); dt = dt or a["dt"]; n = int((T_end or a["horizon"]) / dt)
    t = np.arange(n) * dt; T = np.empty(n); u = np.zeros(n); rng = np.random.default_rng(seed); T[0] = a["T0"]
    for k in range(n - 1):
        reading = T[k] + sensor_sd * rng.normal()
        u[k] = controller(reading) if controller else 0.0
        T[k + 1] = T[k] + dt * (a["power"](t[k]) - (p["hA0"] + p["hA1"] * u[k]) * (T[k] - p["T_amb"])) / p["C"]
    return t, T, u


def watch_asset(asset, fault_at=0.4, loss=0.5, sync=30, seed=RANDOM_STATE):
    """The cooling loses part of its effect. The twin runs its own model and compares it with the reading once per
    window, just before it synchronises again; a cumulative sum of these residuals watches for the fault."""
    rng = np.random.default_rng(seed); dt = asset["dt"]; n = int(asset["horizon"] / dt); t = np.arange(n) * dt
    sd = 0.01 * (asset["limit"] - asset["T_amb"]); duty = lambda T_: float(np.clip((T_ - (asset["limit"] - 6)) / 6, 0, 1))
    T = asset["T0"]; y = T + sd * rng.normal(); model = y; ends, res = [], []
    for k in range(n - 1):
        u = duty(y); h1 = asset["hA1"] * (1 - loss if t[k] >= fault_at * asset["horizon"] else 1)
        T = T + dt * (asset["power"](t[k]) - (asset["hA0"] + h1 * u) * (T - asset["T_amb"])) / asset["C"]
        model = model + dt * (asset["power"](t[k]) - (asset["hA0"] + asset["hA1"] * u) * (model - asset["T_amb"])) / asset["C"]
        y = T + sd * rng.normal()
        if (k + 1) % sync == 0:
            ends.append(t[k + 1]); res.append(y - model); model = y
    ends, res = np.array(ends), np.array(res)
    z = res / res[:int(0.3 * len(res))].std()
    S = np.zeros(len(z))
    for j in range(1, len(z)): S[j] = max(0.0, S[j - 1] + z[j] - 0.5)
    first = lambda mask: float(ends[np.argmax(mask)]) if mask.any() else np.nan
    return ends, z, S, first(S > 5.0), first(np.abs(z) > 4.0), fault_at * asset["horizon"]

asset = ASSETS[ASSET]
t_w, z_w, S_w, alarm_cusum, alarm_limit, t_fault = watch_asset(asset)
unit, scale = ("h", 3600) if asset["horizon"] > 20000 else ("min", 60)
delay = lambda a_: "no alarm" if np.isnan(a_) else f"{(a_ - t_fault) / scale:.1f} {unit} after the fault"
print(f"{asset['name']}: the cooling loses half of its effect at {t_fault / scale:.0f} {unit}")
print(f"   cumulative sum: {delay(alarm_cusum)}; fixed limit on single residuals: {delay(alarm_limit)}")
fig, ax = plt.subplots(2, 1, figsize=(10, 4.6), sharex=True)
ax[0].plot(t_w / scale, z_w, "o-", ms=3, lw=.8, color="#7f8c8d"); ax[0].axhline(4, ls="--", color="#c0392b", lw=1)
ax[0].axhline(-4, ls="--", color="#c0392b", lw=1); ax[0].set_ylabel("residual (SD)")
ax[1].plot(t_w / scale, S_w, "o-", ms=3, color="#2980b9"); ax[1].axhline(5, ls="--", color="#c0392b", lw=1); ax[1].set_ylabel("cumulative sum")
for x in ax: x.axvline(t_fault / scale, color="k", lw=.8)
ax[1].set_xlabel(f"time ({unit})"); plt.tight_layout(); plt.show()

**Exercise D.** How long after the fault does the cumulative sum raise its alarm for your asset? Store the delay in seconds, `alarm_cusum - t_fault`, in `cusum_delay`.

In [ ]:
cusum_delay = None   # your answer

check("Exercise D", cusum_delay, alarm_cusum - t_fault)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
cusum_delay = alarm_cusum - t_fault

check("Exercise D", cusum_delay, alarm_cusum - t_fault)

## 9. Going further (optional): model reduction &#91;[10](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#ref-10)&#93;

*Lecture: [Going further (optional)](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lecture.html#going-further-optional)*

**From the formulas to the code.** `S` is the table of snapshots, with one row per stored state and the four columns SoC (state of charge), $T$, $Q$ and $R$. `Sc` is the standardised table $\mathbf{X}$. `np.linalg.svd` returns the singular values $\sigma_j$ in `sig` and the modes as the rows of `Vt`. `energy = np.cumsum(sig ** 2) / np.sum(sig ** 2)` is the cumulative energy $E_m$ for $m = 1$ to 4. POD in the title of the plot stands for proper orthogonal decomposition.

In [ ]:
# Snapshot matrix: many trajectories under different operating conditions
snaps = []
for T_amb in (20, 28, 35, 42):
    for u in (0.0, 0.4, 0.8):
        d = simulate(CAL, T_end=5400, dt=10.0, cycle="highway", T_amb=float(T_amb),
                     controller=lambda t, x, uu=u: uu)
        snaps.append(d[["SoC", "T", "Q", "R"]].values)
S = np.vstack(snaps)
Sc = (S - S.mean(0)) / S.std(0)
U, sig, Vt = np.linalg.svd(Sc, full_matrices=False)
energy = np.cumsum(sig ** 2) / np.sum(sig ** 2)

print("singular values :", sig.round(2))
print("cumulative energy:", (energy * 100).round(3), "%")
for k in (1, 2, 3, 4):
    print(f"  {k} mode(s): {energy[k-1]*100:6.2f}% of the variance")
print(f"\ntwo modes carry {energy[1]*100:.1f}% of the variance of this four-state system")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].semilogy(range(1, 5), sig, "o-", color="#2980b9")
ax[0].set_xlabel("mode"); ax[0].set_ylabel("singular value"); ax[0].set_title("POD spectrum")
ax[1].plot(range(1, 5), energy * 100, "o-", color="#27ae60")
ax[1].axhline(99.9, ls="--", lw=.8, color="grey")
ax[1].set_xlabel("modes retained"); ax[1].set_ylabel("cumulative energy [%]")
ax[1].set_title("Compressibility of the state")
plt.tight_layout(); plt.show()

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Wrap the Gaussian process in a class that refuses queries outside the design box and falls back to the simulator, and measure the average query time for a mix of inside and outside queries. |
| Electronics and Communication Engineering | Tune the allowance and the decision limit of the cumulative sum so that it raises no false alarm in ten fault-free runs, and report the resulting detection delay. A call `watch_asset(asset, loss=0.0, seed=s)` of section 8 gives a fault-free run and returns the standardised residuals as its second value. |
| Electrical and Electronics Engineering | Add a second residual, from a voltage model, and show whether it detects the rise of the cell resistance that the temperature residual misses. |
| Mechanical Engineering | Replace the linear fade model of section 6 with a model that has a knee and show how the remaining-useful-life estimate and its interval change. |
| Biomedical Engineering | Design a what-if table for a twin of an infusion pump with three scenarios that a clinical engineer would ask about. |
| Civil Engineering | Build a hybrid model for the indoor temperature of a room from a simple physical model and a learned residual, and test it on a hotter month than it was trained on. |
| Aeronautical Engineering | Choose objective weights for the controller that make the fixed rule the winner, and argue whether such weights could be justified. |
| Biotechnology | Train a Gaussian process on simulated yields of a bioreactor and test its predictions outside the temperature range of the design. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-04/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/digital-twin-veltech-NB-lecture/tree/main/days/day-04).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>